# SmartPay AP — Matching Model (D2 Deliverable)

End-to-end invoice reconciliation pipeline:
1. Data ingestion & preprocessing
2. Invoice aggregation
3. Invoice-to-PO matching
4. Mismatch classification
5. Evaluation against labelled ground truth
6. Visualisation

In [ ]:
import os, sys
# Ensure project root is importable
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

DATA_DIR = os.path.join(PROJECT_ROOT, 'data')
print('Project root:', PROJECT_ROOT)
print('Data dir    :', DATA_DIR)

## 1. Data Ingestion

In [ ]:
from src.data_loader import load_invoices, load_po_grn, load_labels, aggregate_invoices

inv_df   = load_invoices(os.path.join(DATA_DIR, 'invoices.csv'))
po_df    = load_po_grn(os.path.join(DATA_DIR, 'po_grn.csv'))
label_df = load_labels(os.path.join(DATA_DIR, 'labelled_mismatches.csv'))

print(f'Invoice line items : {len(inv_df)}')
print(f'PO/GRN records     : {len(po_df)}')
print(f'Labelled mismatches: {len(label_df)}')
inv_df.head(3)

## 2. Aggregation

In [ ]:
inv_agg = aggregate_invoices(inv_df)
print(f'Unique invoices: {len(inv_agg)}')
inv_agg.head(3)

## 3. Invoice-to-PO Matching

In [ ]:
from src.matcher import match_invoices_to_pos

matched = match_invoices_to_pos(inv_agg, po_df)
print(f'Matched  : {matched["matched"].sum()}')
print(f'Unmatched: {(~matched["matched"]).sum()}')
matched.head(5)

## 4. Mismatch Classification

In [ ]:
from src.classifier import classify_mismatches

mismatches = classify_mismatches(matched)
print(f'Total mismatches: {len(mismatches)}')
print(mismatches['mismatch_type'].value_counts().to_string())
mismatches.head(8)

## 5. Evaluation Against Ground Truth

In [ ]:
from src.evaluator import evaluate
import pandas as pd

metrics = evaluate(mismatches, label_df)

rows = []
for cls, m in metrics['per_class'].items():
    rows.append({'Class': cls, **m})
ma = metrics['macro_avg']
rows.append({'Class': 'MACRO AVG', 'precision': ma['precision'],
             'recall': ma['recall'], 'f1': ma['f1'], 'support': ''})

report_df = pd.DataFrame(rows).set_index('Class')
print(report_df.to_string())

## 6. Visualisation

In [ ]:
import matplotlib
matplotlib.use('Agg')  # non-interactive backend for notebooks without display
import matplotlib.pyplot as plt
import numpy as np

# — Mismatch type distribution —
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

type_counts = mismatches['mismatch_type'].value_counts()
axes[0].bar(type_counts.index, type_counts.values, color=['#e74c3c','#3498db','#2ecc71','#f39c12'])
axes[0].set_title('Mismatch Distribution')
axes[0].set_xlabel('Mismatch Type')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=20)

# — F1 scores per class —
classes = list(metrics['per_class'].keys())
f1s = [metrics['per_class'][c]['f1'] for c in classes]
bars = axes[1].bar(classes, f1s, color='#8e44ad')
axes[1].axhline(metrics['macro_avg']['f1'], color='red', linestyle='--', label=f"Macro F1={metrics['macro_avg']['f1']:.2f}")
axes[1].set_title('F1 Score per Class')
axes[1].set_ylim(0, 1.05)
axes[1].set_ylabel('F1 Score')
axes[1].tick_params(axis='x', rotation=20)
axes[1].legend()
for bar, f1 in zip(bars, f1s):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                 f'{f1:.2f}', ha='center', fontsize=10)

plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, 'docs', 'metrics_plot.png'), dpi=120)
plt.show()
print('Plot saved to docs/metrics_plot.png')

## 7. Sample Dispute Emails

In [ ]:
from src.email_generator import generate_dispute_email

# Merge vendor_name into mismatches for emails
vnames = inv_df.groupby('invoice_id')['vendor_name'].first().reset_index()
sample = mismatches.head(3).merge(vnames, on='invoice_id', how='left')

for _, row in sample.iterrows():
    email = generate_dispute_email(row.to_dict(), use_llm=False)
    print('─' * 60)
    print(email)
print('─' * 60)